In [ ]:
import os
from pathlib import Path

import torch
import scipy.io as scio
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib
device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
print('torch device: ' + str(device) + '; torch version: ' + torch.__version__)
import tqdm
matplotlib.rcParams['font.sans-serif'] = ['Arial']
matplotlib.rcParams['font.size'] = 10

plt.rcParams['axes.unicode_minus'] = False 

# 1. Prepare the data and configuration.

## 1.1 Load AKT trajectories for each participant.

In [ ]:
repository_candidates = [Path.cwd(), *Path.cwd().parents]
repository_root = next(
    candidate for candidate in repository_candidates
    if (candidate / "model_code").is_dir()
)
workspace_root = Path(
    os.environ.get("COVERT_READING_WORKSPACE", repository_root / "workspace")
).expanduser().resolve()
clean_data_path = Path(
    os.environ.get("COVERT_READING_DATA_ROOT", workspace_root / "model_data")
).expanduser().resolve()

at_data_path =os.path.join(clean_data_path,"at_dir")
# Define the participant list.
HS_list = []
for HS in HS_list:
    at_data=np.load(os.path.join(at_data_path,f"HS{HS}akt.npy"),allow_pickle=True).item()
    at_data.keys()
    print(HS,": ",np.sum([at_data[key].shape[0] for key in at_data.keys()]))

## 1.2 Load ECoG data, AKT trajectories, and electrode lists for each participant and condition.

In [ ]:

import json
from pathlib import Path
import sys


def _find_model_code_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    candidates += [candidate / "model_code" for candidate in candidates]
    for candidate in candidates:
        if (candidate / "articulatory_movement_synthesizer" / "models" / "helper.py").is_file():
            return candidate
    raise FileNotFoundError("Cannot locate the model_code directory")


model_code_root = _find_model_code_root()
if str(model_code_root) not in sys.path:
    sys.path.insert(0, str(model_code_root))

repository_root = model_code_root.parent
workspace_root = Path(
    os.environ.get("COVERT_READING_WORKSPACE", repository_root / "workspace")
).expanduser().resolve()
electrode_list_path = Path(
    os.environ.get(
        "COVERT_READING_ELECTRODE_LIST",
        workspace_root / "private" / "electrode_lists.json",
    )
).expanduser().resolve()
if not electrode_list_path.is_file():
    raise FileNotFoundError(f"Electrode list file not found: {electrode_list_path}")
with electrode_list_path.open(encoding="utf-8") as handle:
    electrode_lists = json.load(handle)


ecog_data_path = os.path.join(clean_data_path, "HSblockdata")
at_data_path = os.path.join(clean_data_path, "at_dir")


def get_data(HS, elec_type="<PRIVATE_SELECTION>", band="high gamma"):
    if band == "high gamma":
        ecog_data = scio.loadmat(
            os.path.join(
                ecog_data_path,
                f"HS{HS}_Block_overt_covert_70_150_zscore_100Hz.mat",
            )
        )
    elif band == "beta1":
        ecog_data = scio.loadmat(
            os.path.join(
                ecog_data_path,
                f"HS{HS}_Block_overt_covert_12_24_zscore_100Hz.mat",
            )
        )
    elif band == "hgb1":
        ecog_data_hg = scio.loadmat(
            os.path.join(
                ecog_data_path,
                f"HS{HS}_Block_overt_covert_70_150_zscore_100Hz.mat",
            )
        )
        ecog_data_beta = scio.loadmat(
            os.path.join(
                ecog_data_path,
                f"HS{HS}_Block_overt_covert_12_24_zscore_100Hz.mat",
            )
        )
        ecog_data = {}
        for key, high_gamma_value in ecog_data_hg.items():
            if (
                not key.startswith("__")
                and isinstance(high_gamma_value, np.ndarray)
                and key in ecog_data_beta
                and isinstance(ecog_data_beta[key], np.ndarray)
            ):
                ecog_data[key] = np.concatenate(
                    [high_gamma_value, ecog_data_beta[key]], axis=1
                )
            else:
                ecog_data[key] = high_gamma_value
    else:
        raise ValueError(f"Unsupported band: {band}")

    at_data = np.load(
        os.path.join(at_data_path, f"HS{HS}akt.npy"), allow_pickle=True
    ).item()
    elec_list = electrode_lists["movement"][str(HS)][band][elec_type]
    return ecog_data, at_data, elec_list


In [ ]:
for HS in HS_list:
    band='beta1'
    print(HS)
    print("SHARED:", len(get_data(HS,elec_type='SHARED')[2]),
          "only_SASI:", len(get_data(HS,elec_type='only_SASI')[2]),
          "SASI:", len(get_data(HS,elec_type='SASI')[2]),
          "covert_sig:",len(get_data(HS,'covert_sig')[2]),
          "overt_sig:",len(get_data(HS,'overt_sig')[2]),
          "SI-specific:",len(get_data(HS,'SI-specific')[2]),
          "SA-specific:",len(get_data(HS,'SA-specific')[2]))
elec_type_list = ['SHARED','SASI','only_SASI','covert_sig','overt_sig','SI-specific','SA-specific']

## 1.3 Configure the model.

In [ ]:
import torch.nn as nn
from articulatory_movement_synthesizer.models.helper import ResidualBlock, NonLocalBlock, UpSampleBlock, GroupNorm, Swish, DownSampleBlock
import argparse
parser = argparse.ArgumentParser(description="VQGAN Training Script")
parser.add_argument('--latent-dim', type=int, default=256, help='Latent dimension of VQGAN bottleneck (default: 256)')
parser.add_argument('--HS', type=int, default=54, help='Subject number (default: 54)')
parser.add_argument('--time-length', type=int, default=192, help='Input time length for each sample (default: 192)')
parser.add_argument('--num-codebook-vectors', type=int, default=256, help='Number of codebook vectors in VQGAN (default: 256)')
parser.add_argument('--beta', type=float, default=0.25, help='Commitment loss coefficient for VQGAN (default: 0.25)')
parser.add_argument('--image-channels', type=int, default=1, help='Number of channels in input images/traces (default: 1)')
parser.add_argument('--dataset-path', type=Path, default=clean_data_path / 'dataset_for_decoding_trace', help='Path to input dataset')
parser.add_argument('--device', type=str, default="cuda:1", help='CUDA device for training, e.g. "cuda:0"')
parser.add_argument('--batch-size', type=int, default=16, help='Batch size for training (default: 16)')
parser.add_argument('--epochs', type=int, default=80, help='Total number of training epochs (default: 80)')
parser.add_argument('--learning-rate', type=float, default=2.25e-05, help='Learning rate for optimizer (default: 2.25e-5)')
parser.add_argument('--beta1', type=float, default=0.5, help='Adam optimizer beta1 parameter (default: 0.5)')
parser.add_argument('--beta2', type=float, default=0.9, help='Adam optimizer beta2 parameter (default: 0.9)')
parser.add_argument('--disc-start', type=int, default=10000, help='Iteration to start discriminator training (default: 10000)')
parser.add_argument('--disc-factor', type=float, default=1., help='Weighting factor for discriminator loss (default: 1.0)')
parser.add_argument('--rec-loss-factor', type=float, default=1., help='Weight for reconstruction loss (default: 1.0)')
parser.add_argument('--perceptual-loss-factor', type=float, default=1., help='Weight for perceptual loss (default: 1.0)')
parser.add_argument('--lastdim', type=int, default=1, help='Output dimension for encoder (default: 1.0)')
parser.add_argument('--elec_type', type=str, default="all_elecs", help='Electrode type: "all_elecs", "covert_sig", etc.')
parser.add_argument('--reading_name', type=str, default="covert", help='Reading type: "covert" or "overt"')
parser.add_argument('--is-infer', '--is_infer', dest='is_infer', action=argparse.BooleanOptionalAction, default=False, help='Run the held-sound inference experiment')
parser.add_argument('--used_sound', type=str, default='ba', help='Sound used for inference (default: "ba")')
parser.add_argument('--is-fold', '--is_fold', dest='is_fold', action=argparse.BooleanOptionalAction, default=True, help='Run fold-indexed cross-validation')
parser.add_argument('--fold_ind', type=int, default=1, help='Fold index for cross-validation (default: 1)')
parser.add_argument('--band', type=str, default='high gamma', help='Frequency band, e.g. "high gamma"')
parser.add_argument('--is-percentage', '--is_percentage', dest='is_percentage', action=argparse.BooleanOptionalAction, default=False, help='Run the reduced-training-percentage experiment')
parser.add_argument('--percent', type=float, default=1.0, help='the percentage of training data to use (default: 1.0)')


import torch.nn.functional as F



class Encoder(nn.Module):
    def __init__(self, args):
        super(Encoder, self).__init__()
        channels = [128, 128, 128, 256, 256, 512]
        attn_resolutions = [16]
        num_res_blocks = 2
        resolution = 256
        layers = [nn.Conv2d(args.image_channels, channels[0], 3, 1, 1)]
        for i in range(len(channels)-1):
            in_channels = channels[i]
            out_channels = channels[i + 1]
            for j in range(num_res_blocks):
                layers.append(ResidualBlock(in_channels, out_channels))
                in_channels = out_channels
                if resolution in attn_resolutions:
                    layers.append(NonLocalBlock(in_channels))
            if i != len(channels)-2:
                layers.append(DownSampleBlock(channels[i+1]))
                resolution //= 2
        layers.append(ResidualBlock(channels[-1], channels[-1]))
        layers.append(NonLocalBlock(channels[-1]))
        layers.append(ResidualBlock(channels[-1], channels[-1]))
        layers.append(GroupNorm(channels[-1]))
        layers.append(Swish())
        layers.append(nn.Conv2d(channels[-1], args.latent_dim, 3, 1, 1))
        conv_last = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=(1, args.lastdim))
        layers.append(conv_last)
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)
class Decoder(nn.Module):
    def __init__(self, args):
        super(Decoder, self).__init__()
        channels = [512, 256, 256, 128, 128]
        attn_resolutions = [16]
        num_res_blocks = 3
        resolution = 16

        in_channels = channels[0]
        layers = [nn.Conv2d(args.latent_dim, in_channels, 3, 1, 1),
                  ResidualBlock(in_channels, in_channels),
                  NonLocalBlock(in_channels),
                  ResidualBlock(in_channels, in_channels)]

        for i in range(len(channels)):
            out_channels = channels[i]
            for j in range(num_res_blocks):
                layers.append(ResidualBlock(in_channels, out_channels))
                in_channels = out_channels
                if resolution in attn_resolutions:
                    layers.append(NonLocalBlock(in_channels))
            if i != 0:
                layers.append(UpSampleBlock(in_channels))
                resolution *= 2

        layers.append(GroupNorm(in_channels))
        layers.append(Swish())
        layers.append(nn.Conv2d(in_channels, args.image_channels, 3, 1, 1))
                                                                         

    def forward(self, x):
        return self.model(x)

class Codebook(nn.Module):
    def __init__(self, args):
        super(Codebook, self).__init__()
        self.num_codebook_vectors = args.num_codebook_vectors
        self.latent_dim = args.latent_dim
        self.beta = args.beta

        self.embedding = nn.Embedding(self.num_codebook_vectors, self.latent_dim)
        self.embedding.weight.data.uniform_(-1.0 / self.num_codebook_vectors, 1.0 / self.num_codebook_vectors)

    def forward(self, z):
        z = z.permute(0, 2, 3, 1).contiguous()
        z_flattened = z.view(-1, self.latent_dim)

        d = torch.sum(z_flattened**2, dim=1, keepdim=True) + \
            torch.sum(self.embedding.weight**2, dim=1) - \
            2*(torch.matmul(z_flattened, self.embedding.weight.t()))

        min_encoding_indices = torch.argmin(d, dim=1)
        z_q = self.embedding(min_encoding_indices).view(z.shape)

        loss = torch.mean((z_q.detach() - z)**2) + self.beta * torch.mean((z_q - z.detach())**2)

        z_q = z + (z_q - z).detach()

        z_q = z_q.permute(0, 3, 1, 2)

        return z_q, min_encoding_indices, loss
    
    
class linear(nn.Module):
    
    def __init__(self,args):
        super(linear,self).__init__()
        self.layer_1 = nn.Linear(args.latent_dim*15*17,512)
        self.dropout = nn.Dropout(0.4)
        self.layer_2 = nn.Linear(512,32)
        self.relu = nn.LeakyReLU()

    def forward(self, x):
        b,c,t,f = x.shape
        x = x.view(b,c*t*f)
        x = self.layer_2( self.dropout(self.relu (self.layer_1(x))))
        return F.softmax(x)




class Decoder(nn.Module):
    def __init__(self, args):
        super(Decoder, self).__init__()
        channels = [512, 256, 256, 128, 128]
        attn_resolutions = [16]
        num_res_blocks = 3
        resolution = 16

        in_channels = channels[0]
        layers = [nn.Conv2d(args.latent_dim, in_channels, 3, 1, 1),
                  ResidualBlock(in_channels, in_channels),
                  NonLocalBlock(in_channels),
                  ResidualBlock(in_channels, in_channels)]

        for i in range(len(channels)):
            out_channels = channels[i]
            for j in range(num_res_blocks):
                layers.append(ResidualBlock(in_channels, out_channels))
                in_channels = out_channels
                if resolution in attn_resolutions:
                    layers.append(NonLocalBlock(in_channels))
            if i != 0:
                layers.append(UpSampleBlock(in_channels))
                resolution *= 2

        layers.append(GroupNorm(in_channels))
        layers.append(Swish())
        layers.append(nn.Conv2d(in_channels, args.image_channels, 3, 1, 1))
        layers.append(nn.Conv2d(args.image_channels,args.image_channels, kernel_size=(1, 4)))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

class VQGAN(nn.Module):
    def __init__(self, args):
        super(VQGAN, self).__init__()
        self.encoder = Encoder(args).to(device=args.device)
        self.decoder = Decoder(args).to(device=args.device)
        self.codebook = Codebook(args).to(device=args.device)
        self.quant_conv = nn.Conv2d(args.latent_dim, args.latent_dim, 1).to(device=args.device)
        self.post_quant_conv = nn.Conv2d(args.latent_dim, args.latent_dim, 1).to(device=args.device)

    def forward(self, ecogs):
        encoded_images = self.encoder(ecogs)
        quant_conv_encoded_images = self.quant_conv(encoded_images)
        codebook_mapping, codebook_indices, q_loss = self.codebook(quant_conv_encoded_images)
        post_quant_conv_mapping = self.post_quant_conv(codebook_mapping)
        decoded_images = self.decoder(post_quant_conv_mapping)

        return decoded_images, codebook_indices, q_loss

    def encode(self, imgs):
        encoded_images = self.encoder(imgs)
        quant_conv_encoded_images = self.quant_conv(encoded_images)
        codebook_mapping, codebook_indices, q_loss = self.codebook(quant_conv_encoded_images)
        return codebook_mapping, codebook_indices, q_loss

    def decode(self, z):
        post_quant_conv_mapping = self.post_quant_conv(z)
        decoded_images = self.decoder(post_quant_conv_mapping)
        return decoded_images

    def calculate_lambda(self, perceptual_loss, gan_loss):
        last_layer = self.decoder.model[-1]
        last_layer_weight = last_layer.weight
        perceptual_loss_grads = torch.autograd.grad(perceptual_loss, last_layer_weight, retain_graph=True)[0]
        gan_loss_grads = torch.autograd.grad(gan_loss, last_layer_weight, retain_graph=True)[0]

        adaptive_weight = torch.norm(perceptual_loss_grads) / (torch.norm(gan_loss_grads) + 1e-4)
        adaptive_weight = torch.clamp(adaptive_weight, 0, 1e4).detach()
        return 0.8 * adaptive_weight

    @staticmethod
    def adopt_weight(disc_factor, i, threshold, value=0.):
        if i < threshold:
            disc_factor = value
        return disc_factor

    def load_checkpoint(self, path):
        self.load_state_dict(torch.load(path))



In [ ]:
class Encoder(nn.Module):
    def __init__(self, args):
        super(Encoder, self).__init__()
        channels = [64, 64, 128, 128, 256]
        attn_resolutions = [16]
        num_res_blocks = 1
        resolution = 256
        layers = [nn.Conv2d(1, channels[0], 3, 1, 1)]
        for i in range(len(channels)-1):
            in_channels = channels[i]
            out_channels = channels[i + 1]
            for j in range(num_res_blocks):
                layers.append(ResidualBlock(in_channels, out_channels))
                in_channels = out_channels
                if resolution in attn_resolutions:
                    layers.append(NonLocalBlock(in_channels))
            if i != len(channels)-2:
                layers.append(DownSampleBlock(channels[i+1]))
                resolution //= 2
        layers.append(ResidualBlock(channels[-1], channels[-1]))
        layers.append(NonLocalBlock(channels[-1]))
        layers.append(ResidualBlock(channels[-1], channels[-1]))
        layers.append(GroupNorm(channels[-1]))
        layers.append(Swish())
        layers.append(nn.Conv2d(channels[-1], args.latent_dim, 3, 1, 1))
        conv_last = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=(1, args.lastdim))
        layers.append(conv_last)
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

## 1.4 Determine the final encoder kernel size for each participant and electrode list.



In [ ]:
elec_type_list = ['SHARED','SASI','only_SASI','covert_sig','overt_sig','SI-specific','SA-specific','downsample_overt_sig','downsample_covert_sig']
last_dim = {}
# Create the output directory.
os.mkdir(os.path.join(clean_data_path,"dataset_for_decoding_trace/"))
for band in ['high gamma','beta1','hgb1']:
      for elec_type in elec_type_list:
            last_dim[elec_type] = {}
            for HS in HS_list:
                  args = parser.parse_args(args=[])
                  _,_,elec_list = get_data(HS,elec_type=elec_type,band=band)
                  if len(elec_list)<16:
                        print(f"{elec_type} HS{HS} has {len(elec_list)} electrodes, skip it.")
                        continue
                  print(len(elec_list))

                  x = torch.randn(16,1,192,int(len(elec_list)))
                  encoder = Encoder(args)
                  decoder = Decoder(args)
                  latentz  =encoder(x)
                  print(    latentz.shape[-1])
                  last_dim[elec_type][f"HS{HS}"] =  latentz.shape[-1]
                  args.lastdim = latentz.shape[-1]
                  encoder = Encoder(args)
                  print(    latentz.shape)
                  latentz  =encoder(x)
                  y = decoder(latentz)
                  print(y.shape)
      np.save(os.path.join(clean_data_path,"dataset_for_decoding_trace",f"last_dim_{band}.npy"),last_dim,allow_pickle=True)


# 2. Generate the datasets.

In [ ]:
data_save_path =os.path.join(clean_data_path,"dataset_for_decoding_trace")
from os import read
from numpy import dtype
import torch.utils.data as data
from sklearn.model_selection import train_test_split,StratifiedShuffleSplit
import dill
class CustomDataset(data.Dataset):
    def __init__(self, ecog_dict,at_dict,elec_list,reading_name,elec_type,percentage=1):
        self.data = []
        self.labels = []
        self.sound_labels = []
        i=0
        for key, value in at_dict.items():
            
            data_length = min(len(np.swapaxes(ecog_dict[f"ECoG_{reading_name}_{key}"][:,:,104:296],1,2)),len(value))
            data_length_original = data_length
            np.random.seed(2)
            # Randomly select the requested fraction of the original samples.
            data_length = int(data_length * percentage)
            # Generate a shuffled sample-index array.
            indices = np.random.permutation(data_length_original)[:data_length]
            if elec_type=="all_elecs":
                self.data.extend(np.swapaxes(ecog_dict[f"ECoG_{reading_name}_{key}"][:,:,104:296],1,2)[indices])  # Append samples from the source dictionary.
                self.labels.extend(value[indices,4:196])
            
            else:
                self.data.extend(np.swapaxes(ecog_dict[f"ECoG_{reading_name}_{key}"][:,elec_list,104:296],1,2)[indices])  # Append samples from the source dictionary.
                self.labels.extend(value[indices,4:196])
            
            self.sound_labels.extend([i]*data_length)
            i+=1

        self.data =np.array(self.data)
        # Pad the data to a fixed width of 16 channels.
        pad_size = 16 - self.data.shape[2]
        if pad_size > 0:
            pad_width = ((0, 0), (0, 0), (0, pad_size))
            padded_array = np.pad(self.data, pad_width, mode='constant', constant_values=0)
            self.data = padded_array
            print(f"Data padded to shape: {self.data.shape}")
        self.labels = np.array(self.labels)
        self.sound_labels = np.array(self.sound_labels)
        self.pitchs =  self.labels[:,:,[-1]]


    def __len__(self):
        return len(self.data)
    def get_label(self):

        return self.sound_labels

    def __getitem__(self, idx):
        x = self.data[idx]
        y = self.labels[idx,:]
        label = self.sound_labels[idx]
        #,torch.tensor(label,dtype=torch.long),torch.tensor(label,dtype=torch.long)
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32),torch.tensor(label,dtype=torch.long)
    


## 2.1 Load the data.

In [ ]:
import os
import dill
import numpy as np
from torch.utils.data import DataLoader, Subset, Dataset
from sklearn.model_selection import StratifiedKFold
from typing import Tuple, List, Optional
import torch

def generate_dataset_fold(
    HS: int,
    reading_name: str,
    elec_type: str = "all_elecs",
    batch_size: int = 16,
    random_state: int = 2,
    save_path: Optional[str] = None,
    is_token_percent:bool=False
) -> Tuple[List[DataLoader], List[DataLoader]]:
    """
    Create DataLoaders for five-fold cross-validation.
    
    Args:
        HS: participant identifier
        reading_name: reading condition
        elec_type: electrode-selection type
        batch_size: number of samples per batch
        random_state: random seed
        save_path: optional output directory; no files are saved when None
    
    Returns:
        train_loaders: training DataLoaders
        val_loaders: validation DataLoaders
    """
    
    # 1. Load participant data.
    ecog_data, at_data, elec_list = get_data(HS, elec_type,band=band)
    
    if is_token_percent:
        for percent in [0.2,0.4,0.6,0.8]:
                    
            # 2. Create the dataset.
            dataset = CustomDataset(ecog_data, at_data, elec_list, reading_name, elec_type,percentage=percent)
            
            # 3. Set random seeds for reproducibility.
            np.random.seed(random_state)
            torch.manual_seed(random_state)  # Set the PyTorch seed.
            
            # 4. Create stratified K-fold splits.
            kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
            
            # 5. Read labels without materializing the full dataset.
            # Access labels by index to limit memory use.
            labels = []
            for i in range(len(dataset)):
                _, _, label = dataset[i]  # Read only the label.
                labels.append(label)
            
            y = np.array(labels)
            indices = np.arange(len(dataset))
            
            train_loaders = []
            val_loaders = []
            
            # 6. Build DataLoaders for each fold.
            for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(indices, y)):
                #print(f"Generating Fold {fold_idx + 1}/5...")
                
                # Create fold subsets.
                train_subset = Subset(dataset, train_idx)
                val_subset = Subset(dataset, val_idx)
                
                # Create DataLoaders.
                train_loader = DataLoader(
                    train_subset, 
                    batch_size=batch_size, 
                    shuffle=True,
                    num_workers=4,  # Use worker processes for loading.
                    pin_memory=True  # Pin host memory for GPU transfer.
                )
                
                val_loader = DataLoader(
                    val_subset, 
                    batch_size=batch_size, 
                    shuffle=False,
                    num_workers=4,
                    pin_memory=True
                )
                
                # Append the loaders.
                train_loaders.append(train_loader)
                val_loaders.append(val_loader)
                
                # 7. Optionally serialize the loaders.
                if save_path:
                    os.makedirs(save_path, exist_ok=True)
                    
                    train_filename = f"HS{HS}_{reading_name}_train_loader_{band}_{elec_type}_fold{fold_idx}_percentage_{percent}.pkl"
                    val_filename = f"HS{HS}_{reading_name}_val_loader_{band}_{elec_type}_fold{fold_idx}_percentage_{percent}.pkl"
                    
                    with open(os.path.join(save_path, train_filename), 'wb') as f:
                        dill.dump(train_loader, f)
                    
                    with open(os.path.join(save_path, val_filename), 'wb') as f:
                        dill.dump(val_loader, f)
            
                    #print(f"Successfully generated {len(train_loaders)} folds for {HS} {reading_name} {elec_type}")

    else:
        # 2. Create the dataset.
            dataset = CustomDataset(ecog_data, at_data, elec_list, reading_name, elec_type,percentage=1)
            
            # 3. Set random seeds for reproducibility.
            np.random.seed(random_state)
            torch.manual_seed(random_state)  # Set the PyTorch seed.
            
            # 4. Create stratified K-fold splits.
            kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
            
            # 5. Read labels without materializing the full dataset.
            # Access labels by index to limit memory use.
            labels = []
            for i in range(len(dataset)):
                _, _, label = dataset[i]  # Read only the label.
                labels.append(label)
            
            y = np.array(labels)
            indices = np.arange(len(dataset))
            
            train_loaders = []
            val_loaders = []
            
            # 6. Build DataLoaders for each fold.
            for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(indices, y)):
                #print(f"Generating Fold {fold_idx + 1}/5...")
                
                # Create fold subsets.
                train_subset = Subset(dataset, train_idx)
                val_subset = Subset(dataset, val_idx)
                
                # Create DataLoaders.
                train_loader = DataLoader(
                    train_subset, 
                    batch_size=batch_size, 
                    shuffle=True,
                    num_workers=4,  # Use worker processes for loading.
                    pin_memory=True  # Pin host memory for GPU transfer.
                )
                
                val_loader = DataLoader(
                    val_subset, 
                    batch_size=batch_size, 
                    shuffle=False,
                    num_workers=4,
                    pin_memory=True
                )
                
                # Append the loaders.
                train_loaders.append(train_loader)
                val_loaders.append(val_loader)
                
                # 7. Optionally serialize the loaders.
                if save_path:
                    os.makedirs(save_path, exist_ok=True)
                    
                    train_filename = f"HS{HS}_{reading_name}_train_loader_{band}_{elec_type}_fold{fold_idx}.pkl"
                    val_filename = f"HS{HS}_{reading_name}_val_loader_{band}_{elec_type}_fold{fold_idx}.pkl"
                    
                    with open(os.path.join(save_path, train_filename), 'wb') as f:
                        dill.dump(train_loader, f)
                    
                    with open(os.path.join(save_path, val_filename), 'wb') as f:
                        dill.dump(val_loader, f)
            
                    #print(f"Successfully generated {len(train_loaders)} folds for {HS} {reading_name} {elec_type}")

    return train_loaders, val_loaders



## 2.2 Generate five-fold cross-validation datasets.

In [ ]:
import tqdm
elec_type_list = ['all_elecs','downsampled_all_elecs','SHARED','SASI','only_SASI','covert_sig',
                  'overt_sig','SI-specific','SA-specific','downsample_overt_sig','downsample_covert_sig']
for band in ['beta1','high gamma','hgb1']:
    last_dim = np.load(f"{data_save_path}/last_dim_{band}.npy",allow_pickle=True).item()  
    for reading_name in ["overt",'covert']:
        for elec_type in elec_type_list:
            
            if reading_name=='overt' and elec_type in ["covert_sig",'downsample_covert_sig','SI-specific']:
                continue
            if reading_name=='covert' and elec_type in ["overt_sig",'downsample_overt_sig','SA-specific']:
                continue    
            for HS in tqdm.tqdm(HS_list,desc=f"Processing {reading_name} {elec_type}"):
                generate_dataset_fold(HS,reading_name=reading_name,elec_type=elec_type,save_path=data_save_path,is_token_percent=False)

# 3. Save model predictions and ground-truth test data.

In [ ]:
from scipy.stats import pearsonr
import numpy as np
import gc

os.makedirs(os.path.join(clean_data_path,"decoded_result"), exist_ok=True)

def get_predicted_trace(args,vqgan,test_loader,test_name,infer=False,perm=False,percent=1.0):
    
    _,at_data,_ = get_data(args.HS,args.elec_type)

    r_set = {"real_trace":{list(at_data.keys())[i]:[] for i in range(len(list(at_data.keys())))},
             "predicted_trace":{list(at_data.keys())[i]:[] for i in range(len(list(at_data.keys())))}}

    for ecogs_trace in  test_loader:
        ecogs = ecogs_trace[0].unsqueeze(1).to(device=args.device)
        trace = ecogs_trace[1].unsqueeze(1).to(device=args.device)
        label = ecogs_trace[2]

        decoded_images, _, q_loss = vqgan(ecogs)
        for i in range(len(ecogs)):
            r_set["real_trace"][list(at_data.keys())[label[i]-1]].append(trace[i].cpu().detach().numpy())
            r_set["predicted_trace"][list(at_data.keys())[label[i]-1]].append(decoded_images[i].cpu().detach().numpy())
    del ecogs, trace
    gc.collect()
    for key in list(r_set.keys()):
        for sound in list(r_set[key].keys()):
            r_set[key][sound] = np.array( r_set[key][sound])
    npy_save_name_prefix = f"HS{args.HS}_{args.reading_name}_on_{test_name}_{args.elec_type}"
    npy_data_save_path =os.path.join(clean_data_path,"decoded_result")
    if perm:
        if infer:
            npy_save_name = f"{npy_save_name_prefix}_{args.used_sound}_{args.band}_perm.npy"
        elif args.is_percentage:
            npy_save_name = f"{npy_save_name_prefix}_fold{args.fold_ind}_{args.band}_percentage_{percent}_perm.npy"
        else:   
            npy_save_name = f"{npy_save_name_prefix}_fold{args.fold_ind}_{args.band}_perm.npy"
    else:
        if infer:
            npy_save_name = f"{npy_save_name_prefix}_{args.used_sound}_{args.band}.npy"
        elif args.is_percentage:
            npy_save_name = f"{npy_save_name_prefix}_fold{args.fold_ind}_{args.band}_percentage_{percent}.npy"
        else:
            npy_save_name = f"{npy_save_name_prefix}_fold{args.fold_ind}_{args.band}.npy"

    np.save(os.path.join(npy_data_save_path,npy_save_name),r_set,allow_pickle=True)
    return 0
def check_complete(args,test_name,perm,infer,percent=1.0):
    npy_save_name_prefix = f"HS{args.HS}_{args.reading_name}_on_{test_name}_{args.elec_type}"
    npy_data_save_path =os.path.join(clean_data_path,"decoded_result")
    if perm:
        if infer:
            npy_save_name = f"{npy_save_name_prefix}_{args.used_sound}_{args.band}_perm.npy"
        elif args.is_percentage:
            npy_save_name = f"{npy_save_name_prefix}_fold{args.fold_ind}_{args.band}_percentage_{percent}_perm.npy"
        else:
            npy_save_name = f"{npy_save_name_prefix}_fold{args.fold_ind}_{args.band}_perm.npy"
    else:
        if infer:
            npy_save_name = f"{npy_save_name_prefix}_{args.used_sound}_{args.band}.npy"
        elif args.is_percentage:
            npy_save_name = f"{npy_save_name_prefix}_fold{args.fold_ind}_{args.band}_percentage_{percent}.npy"
        else:
            npy_save_name = f"{npy_save_name_prefix}_fold{args.fold_ind}_{args.band}.npy"

    if os.path.exists(os.path.join(npy_data_save_path,npy_save_name)):
        return True
    else:
        return False

## 3.1 Run five-fold cross-validation.

In [ ]:
args = parser.parse_args(args=[])
band = 'hgb1'
npy_data_save_path =os.path.join(clean_data_path,"decoded_result")
elec_type_list = ["downsampled_all_elecs",'SHARED','SASI','only_SASI','covert_sig','overt_sig','SI-specific','SA-specific']
elec_type_cross_modality = ['SHARED','SASI','only_SASI']
for band in ['beta1','high gamma','hgb1']:
    args.band = band
    last_dim = np.load(f"{data_save_path}/last_dim_{band}.npy",allow_pickle=True).item()

    for reading_name in ["overt",'covert']:
        for elec_type in elec_type_list:
            HS_list =  [45,47,48,50,54,71,73,76,78]
            args.reading_name=reading_name
            args.elec_type = elec_type

            if args.reading_name=='overt' and args.elec_type in ["covert_sig",'downsample_covert_sig','SI-specific']:
                continue 
            elif args.reading_name=='covert' and args.elec_type in ["overt_sig",'downsample_overt_sig','SA-specific']:
                continue

            else:
                for HS in HS_list:
                    args.HS = HS
                    args.is_infer = False
                    if args.elec_type in last_dim:
                        if f'HS{HS}' not in last_dim[args.elec_type]:
                            print(f"HS{HS} has no {args.elec_type} electrodes, pad it.")
                        
                            args.lastdim = 1
                        else:
                            args.lastdim = last_dim[args.elec_type][f"HS{args.HS}"]

                        args.is_fold = True
                        for fold_ind in range(5):
                            args.fold_ind = fold_ind
                            test_name = reading_name
                            npy_save_name_prefix = f"HS{args.HS}_{args.reading_name}_on_{test_name}_{args.elec_type}"

                            check_complete_flag = check_complete(args,test_name,perm=True,infer=False)
                            if not check_complete_flag:
                                print(f"Processing {npy_save_name_prefix}_fold{args.fold_ind}_{args.band}_perm.npy")
                                vqgan = VQGAN(args)

                                # Evaluate the validation set without loading a trained checkpoint.
                                vqgan.eval()
                                val_filename = f"HS{args.HS}_{args.reading_name}_val_loader_{band}_{args.elec_type}_fold{args.fold_ind}.pkl"
                                with open(os.path.join(data_save_path, val_filename), 'rb') as f:
                                    val_loader = dill.load(f)

                                get_predicted_trace(args,vqgan,val_loader,test_name=reading_name,infer=False,perm=True)
                            else:
                                print(f"{npy_save_name_prefix}_fold{args.fold_ind}_{args.band}_perm.npy exists, skip it.")
                                           
                            check_complete_flag = check_complete(args,test_name,perm=False,infer=False)
                            # Load the trained checkpoint.
                            if check_complete_flag:
                                print(f"{npy_save_name_prefix}_fold{args.fold_ind}_{args.band}.npy exists, skip it.")
                                continue
                            vqgan = VQGAN(args)
                            checkpoints_path=os.path.join(clean_data_path,f'checkpoints/vqgan{args.HS}_epoch_79_{args.reading_name}_{args.elec_type}_{band}_{args.fold_ind}.pt')
                            if not os.path.exists(checkpoints_path):
                                print(f"{checkpoints_path} not exists, skip it.")
                                continue
                            vqgan.load_checkpoint(checkpoints_path)
                            vqgan.eval()
                            val_filename = f"HS{args.HS}_{args.reading_name}_val_loader_{band}_{args.elec_type}_fold{args.fold_ind}.pkl"
                            with open(os.path.join(data_save_path, val_filename), 'rb') as f:
                                val_loader = dill.load(f)
                            get_predicted_trace(args,vqgan,val_loader,test_name=reading_name,infer=False,perm=False)
                            

## 3.2 Run cross-modality evaluation.

In [ ]:
for band in ['beta1','high gamma','hgb1']:
    args.band = band
    last_dim = np.load(f"{data_save_path}/last_dim_{band}.npy",allow_pickle=True).item()

    for reading_name in ["overt",'covert']:
        for elec_type in elec_type_cross_modality:
            HS_list =  [45,47,48,50,54,71,73,76,78]
            args.reading_name=reading_name
            args.elec_type = elec_type

            if args.reading_name=='overt' and args.elec_type in ["covert_sig"]:
                continue
            elif args.reading_name=='covert' and args.elec_type in ["overt_sig"]:
                continue
            if args.reading_name=='overt' and args.elec_type in ["SI-specific"]:
                continue
            elif args.reading_name=='covert' and args.elec_type in ["SA-specific"]:
                continue
            else:
                for HS in HS_list:
                    args.HS = HS
                    args.is_infer = False
                    if args.elec_type in last_dim:
                        if f'HS{HS}' not in last_dim[args.elec_type]:
                            print(f"HS{HS} has no {args.elec_type} electrodes, skip it.")
                            continue
                    if args.elec_type == "all_elecs":
                        args.lastdim = 32 if band == 'hgb1' else 16
                    elif args.elec_type == "downsampled_all_elecs":
                        args.lastdim = 8 if band == 'hgb1' else 4
                    else:
                        args.lastdim = last_dim[f'{args.elec_type}'][f"HS{args.HS}"]
                    args.is_fold = True
                    test_name = 'covert' if reading_name == 'overt' else 'overt'

                    for fold_ind in range(5):
                        args.fold_ind = fold_ind
                        # if os.path.exists(f"{npy_data_save_path}/HS{args.HS}_{args.reading_name}_on_{args.reading_name}_{args.elec_type}_fold{args.fold_ind}.npy"):
                        #     continue
                        print(f"HS{args.HS}_{args.reading_name}_{args.elec_type}_{args.fold_ind}, last dim:{args.lastdim}")
                        vqgan = VQGAN(args)
                       
                        vqgan.eval()
                        val_filename = f"HS{args.HS}_{test_name}_val_loader_{band}_{args.elec_type}_fold{args.fold_ind}.pkl"
                        with open(os.path.join(data_save_path, val_filename), 'rb') as f:
                            val_loader = dill.load(f)

                        # Evaluate on the other modality.
                        get_predicted_trace(args,vqgan,val_loader,test_name=test_name,infer=False,perm=True)

                        # Load the trained checkpoint.
                        checkpoints_path=os.path.join(clean_data_path,f'checkpoints/vqgan{args.HS}_epoch_79_{args.reading_name}_{args.elec_type}_{band}_{args.fold_ind}.pt')
                        if not os.path.exists(checkpoints_path):
                            print(f"{checkpoints_path} not exists, skip it.")
                            continue
                        vqgan.load_checkpoint(checkpoints_path)
                        vqgan.eval()
                        val_filename = f"HS{args.HS}_{test_name}_val_loader_{band}_{args.elec_type}_fold{args.fold_ind}.pkl"
                        with open(os.path.join(data_save_path, val_filename), 'rb') as f:
                            val_loader = dill.load(f)
                        get_predicted_trace(args,vqgan,val_loader,test_name=test_name,infer=False,perm=False)


# 4. Run inference for held-out conditions.

## 4.1 Configure the dataset.

In [ ]:
from os import read
from matplotlib import contour
from numpy import dtype
import torch.utils.data as data
from sklearn.model_selection import train_test_split,StratifiedShuffleSplit
import dill
class CustomDataset(data.Dataset):
    def __init__(self, ecog_dict,at_dict,elec_list,reading_name,elec_type,used_key=['ba','da','ga','pa','ka','ta','sha']):
        self.data = []
        self.labels = []
        self.sound_labels = []
        i=0
        for key, value in at_dict.items():
            if key in used_key:
                    
                data_length = min(len(np.swapaxes(ecog_dict[f"ECoG_{reading_name}_{key}"][:,:,104:296],1,2)),len(value))
                
                if elec_type=="all_elecs":
                    self.data.extend(np.swapaxes(ecog_dict[f"ECoG_{reading_name}_{key}"][:,:,104:296],1,2)[:data_length])  # Append samples from the source dictionary.
                    self.labels.extend(value[:data_length,4:196])
                
                else:
                    self.data.extend(np.swapaxes(ecog_dict[f"ECoG_{reading_name}_{key}"][:,elec_list,104:296],1,2)[:data_length])  # Append samples from the source dictionary.
                    self.labels.extend(value[:data_length,4:196])
                i+=1
                self.sound_labels.extend([i]*data_length)

        self.data =np.array(self.data)
        pad_size = 16 - self.data.shape[2]
        if pad_size > 0:
            pad_width = ((0, 0), (0, 0), (0, pad_size))
            padded_array = np.pad(self.data, pad_width, mode='constant', constant_values=0)
            self.data = padded_array
            print(f"Data padded to shape: {self.data.shape}")
        self.labels = np.array(self.labels)
        self.sound_labels = np.array(self.sound_labels)
        self.pitchs =  self.labels[:,:,[-1]]


    def __len__(self):
        return len(self.data)
    def get_label(self):

        return self.sound_labels

    def __getitem__(self, idx):
        x = self.data[idx]
        y = self.labels[idx,:]
        label = self.sound_labels[idx]
        #,torch.tensor(label,dtype=torch.long),torch.tensor(label,dtype=torch.long)
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32),torch.tensor(label,dtype=torch.long)
    


## 4.2 Configure the data loaders.

In [ ]:

def generate_dataset_infer(HS,reading_name,elec_type="all_elecs",save_path=data_save_path,band='beta1'):
    ecog_data,at_data,elec_list = get_data(HS,elec_type=elec_type,band=band)
    print(f"HS{HS} {reading_name} {elec_type} has {len(elec_list)} electrodes.")
    if HS < 70:
        all_sound_list =['ba','da','ga','bu','du','gu']
        two_sound_list = [('ba','bu'),('da','du'),('ga','gu')]
        tri_sound_list = [('ba','da','ga'),('bu','du','gu')]
    else:
        all_sound_list = ['ba','da','ga','pa','ka','ta','sha','sa']
        two_sound_list = [('ba','pa'),('da','ta'),('ga','ka'),('sha','sa')]
        tri_sound_list = [('ba','da','ga'),('pa','ta','ka')]

    for sound_used in all_sound_list:
        val_sound_list = [sound_used]
        train_sound_list = [i for i in all_sound_list if i !=sound_used] 

        train_subset = CustomDataset(ecog_data,at_data,elec_list,reading_name,elec_type,train_sound_list)
        val_subset =  CustomDataset(ecog_data,at_data,elec_list,reading_name,elec_type,val_sound_list)


        train_loader = data.DataLoader(train_subset, batch_size=16, shuffle=True)
        val_loader = data.DataLoader(val_subset, batch_size=16, shuffle=False)
        
        with open(os.path.join(data_save_path,f"HS{HS}_{reading_name}_train_loader_{band}_{elec_type}_infer_{sound_used}.pkl"),'wb') as f:
            dill.dump(train_loader,f)
        with open(os.path.join(data_save_path,f"HS{HS}_{reading_name}_val_loader_{band}_{elec_type}_infer_{sound_used}.pkl"),'wb') as f:
            dill.dump(val_loader,f)
    # Two-syllable held-out condition.
    for sound_used in two_sound_list:
        val_sound_list = list(sound_used)
        train_sound_list = [i for i in all_sound_list if i not in sound_used]

        train_subset = CustomDataset(ecog_data,at_data,elec_list,reading_name,elec_type,train_sound_list)
        val_subset =  CustomDataset(ecog_data,at_data,elec_list,reading_name,elec_type,val_sound_list)

        train_loader = data.DataLoader(train_subset, batch_size=16, shuffle=True)
        val_loader = data.DataLoader(val_subset, batch_size=16, shuffle=False)
        
        with open(os.path.join(data_save_path,f"HS{HS}_{reading_name}_train_loader_{band}_{elec_type}_infer_{sound_used}.pkl"),'wb') as f:
            dill.dump(train_loader,f)
        with open(os.path.join(data_save_path,f"HS{HS}_{reading_name}_val_loader_{band}_{elec_type}_infer_{sound_used}.pkl"),'wb') as f:
            dill.dump(val_loader,f)

    # Three-syllable held-out condition.
    for sound_used in tri_sound_list:   
        val_sound_list = list(sound_used)
        train_sound_list = [i for i in all_sound_list if i not in sound_used]

        train_subset = CustomDataset(ecog_data,at_data,elec_list,reading_name,elec_type,train_sound_list)
        val_subset =  CustomDataset(ecog_data,at_data,elec_list,reading_name,elec_type,val_sound_list)


        train_loader = data.DataLoader(train_subset, batch_size=16, shuffle=True)
        val_loader = data.DataLoader(val_subset, batch_size=16, shuffle=False)
        
        with open(os.path.join(data_save_path,f"HS{HS}_{reading_name}_train_loader_{band}_{elec_type}_infer_{sound_used}.pkl"),'wb') as f:
            dill.dump(train_loader,f)
        with open(os.path.join(data_save_path,f"HS{HS}_{reading_name}_val_loader_{band}_{elec_type}_infer_{sound_used}.pkl"),'wb') as f:
            dill.dump(val_loader,f)


    return 0


## 4.3 Build the datasets.

In [ ]:
import tqdm
all_test_loader = {}
elec_type_list = ['SHARED','SASI','only_SASI','covert_sig','overt_sig']
band_list = ['beta1','high gamma','hgb1']
for band in band_list:
    last_dim = np.load(f"{data_save_path}/last_dim_{band}.npy",allow_pickle=True).item()  
    for reading_name in ["overt",'covert']:
        for elec_type in elec_type_list:
            HS_list =  [45,47,48,50,54,71,73,76,78]
            if reading_name=='overt' and elec_type in ["covert_sig"]:
                continue
            if reading_name=='covert' and elec_type in ["overt_sig"]:
                continue    
            for HS in tqdm.tqdm(HS_list,desc=f"Processing {reading_name} {elec_type}"):
               
                generate_dataset_infer(HS,reading_name=reading_name,elec_type=elec_type,save_path=data_save_path,band=band)
                    

## 4.4 Save the outputs.

In [ ]:
import tqdm
all_test_loader = {}
elec_type_list = ['covert_sig','overt_sig']
band_list = ['high gamma']
args.is_infer = True
for band in band_list:
    last_dim = np.load(f"{data_save_path}/last_dim_{band}.npy",allow_pickle=True).item()  
    args.band = band
    for reading_name in ["overt",'covert']:
        args.reading_name=reading_name
        for elec_type in elec_type_list:
            args.elec_type = elec_type
            HS_list =  [45,47,48,50,54,71,73,76,78]
            args.reading_name=reading_name
            args.elec_type = elec_type

            if args.reading_name=='overt' and args.elec_type in ["covert_sig"]:
                continue
            elif args.reading_name=='covert' and args.elec_type in ["overt_sig"]:
                continue
            if args.reading_name=='overt' and args.elec_type in ["SI-specific"]:
                continue
            elif args.reading_name=='covert' and args.elec_type in ["SA-specific"]:
                continue
            test_name = reading_name 
            for HS in tqdm.tqdm(HS_list,desc=f"Processing {reading_name} {elec_type}"):
                args.HS = HS
                if args.elec_type in last_dim:
                    if f'HS{HS}' not in last_dim[args.elec_type]:
                        print(f"HS{HS} has no {args.elec_type} electrodes, padding it.")
                        args.lastdim = 1
                    else:
                        if args.elec_type == "all_elecs":
                            args.lastdim = 32 if band == 'hgb1' else 16
                        elif args.elec_type == "downsampled_all_elecs":
                            args.lastdim = 8 if band == 'hgb1' else 4
                        else:
                            args.lastdim = last_dim[f'{args.elec_type}'][f"HS{args.HS}"]
                if HS < 70:
                        all_sound_list =['ba','da','ga','bu','du','gu']
                        two_sound_list = [('ba','bu'),('da','du'),('ga','gu')]
                        tri_sound_list = [('ba','da','ga'),('bu','du','gu')]
                else:
                    all_sound_list = ['ba','da','ga','pa','ka','ta','sha','sa']
                    two_sound_list = [('ba','pa'),('da','ta'),('ga','ka'),('sha','sa')]
                    tri_sound_list = [('ba','da','ga'),('pa','ta','ka')]

                for sound_used in all_sound_list+two_sound_list+tri_sound_list:
                    args.HS = HS
                    args.used_sound = sound_used
                    val_filename = f"HS{HS}_{reading_name}_val_loader_{band}_{elec_type}_infer_{sound_used}.pkl"
                    with open(os.path.join(data_save_path, val_filename), 'rb') as f:
                        val_loader = dill.load(f)
                    print(f"HS{HS} {reading_name} {elec_type} has {len(elec_list)} electrodes.")
                    vqgan = VQGAN(args)
                   
                    vqgan.eval()

                    get_predicted_trace(args,vqgan,val_loader,test_name=test_name,infer=True,perm=True)

                    # Load the trained checkpoint.
                    checkpoints_path=os.path.join(clean_data_path,f'checkpoints/vqgan{args.HS}_epoch_79_{args.reading_name}_{args.elec_type}_{band}_infer_{args.used_sound}.pt')
                    if not os.path.exists(checkpoints_path):
                        print(f"{checkpoints_path} not exists, skip it.")
                        continue
                    vqgan.load_checkpoint(checkpoints_path)
                    vqgan.eval()

                    get_predicted_trace(args,vqgan,val_loader,test_name=test_name,infer=True,perm=False)

# 5. Evaluate reduced-training-data conditions.

In [ ]:
args = parser.parse_args(args=[])
band = 'hgb1'

elec_type_list = ["downsampled_all_elecs",'SHARED','SASI','only_SASI','covert_sig','overt_sig','SI-specific','SA-specific']
elec_type_cross_modality = ['covert_sig','SASI','only_SASI']
for band in ['beta1','high gamma','hgb1']:
    args.band = band
    last_dim = np.load(f"{data_save_path}/last_dim_{band}.npy",allow_pickle=True).item()

    for reading_name in ["overt",'covert']:
        for elec_type in ['covert_sig','overt_sig']:
            args.reading_name=reading_name
            args.elec_type = elec_type

            if args.reading_name=='overt' and args.elec_type in ["covert_sig"]:
                continue
            elif args.reading_name=='covert' and args.elec_type in ["overt_sig"]:
                continue
            if args.reading_name=='overt' and args.elec_type in ["SI-specific"]:
                continue
            elif args.reading_name=='covert' and args.elec_type in ["SA-specific"]:
                continue
            else:
                for HS in HS_list:
                    args.HS = HS
                    args.is_infer = False
                    if args.elec_type in last_dim:
                        if f'HS{HS}' not in last_dim[args.elec_type]:
                            print(f"HS{HS} has no {args.elec_type} electrodes, skip it.")
                            continue
                    if args.elec_type == "all_elecs":
                        args.lastdim = 32 if band=='hgb1' else 16
                    elif args.elec_type == "downsampled_all_elecs":
                        args.lastdim = 8 if band=='hgb1' else 4
                    else:
                        args.lastdim = last_dim[f'{args.elec_type}'][f"HS{args.HS}"]
                    args.is_fold = True
                    args.is_percentage = True
                    for percent in [0.2,0.4,0.6,0.8]:
                        args.percent = percent
                        for fold_ind in range(5):
                            args.fold_ind = fold_ind
                            test_name = reading_name
                            npy_save_name_prefix = f"HS{args.HS}_{args.reading_name}_on_{test_name}_{args.elec_type}"

                            check_complete_flag = check_complete(args,test_name,perm=True,infer=False,percent=percent)
                            if not check_complete_flag:
                                print(f"Processing {npy_save_name_prefix}_fold{args.fold_ind}_{args.band}_perm.npy")
                                vqgan = VQGAN(args)

                                # Evaluate the validation set without loading a trained checkpoint.
                                vqgan.eval()
                                val_filename = f"HS{args.HS}_{args.reading_name}_val_loader_{band}_{args.elec_type}_fold{args.fold_ind}_percentage_{args.percent}.pkl"
                                with open(os.path.join(data_save_path, val_filename), 'rb') as f:
                                    val_loader = dill.load(f)
                                # Check current GPU memory usage.

                                get_predicted_trace(args,vqgan,val_loader,test_name=reading_name,infer=False,perm=True,percent=percent)
                            else:
                                print(f"{npy_save_name_prefix}_fold{args.fold_ind}_{args.band}_perm.npy exists, skip it.")
                            # Evaluate the validation set with the trained checkpoint.
                            check_complete_flag = check_complete(args,test_name,perm=False,infer=False,percent=percent)
                            # Load the trained checkpoint.
                            if check_complete_flag:
                                print(f"{npy_save_name_prefix}_fold{args.fold_ind}_{args.band}.npy exists, skip it.")
                                continue
                            vqgan = VQGAN(args)
                            checkpoints_path=os.path.join(clean_data_path,f'checkpoints/vqgan{args.HS}_epoch_79_{args.reading_name}_{args.elec_type}_{band}_{args.fold_ind}_percentage_{percent}.pt')
                            if not os.path.exists(checkpoints_path):
                                print(f"{checkpoints_path} not exists, skip it.")
                                continue
                            vqgan.load_checkpoint(checkpoints_path)
                            vqgan.eval()
                            val_filename = f"HS{args.HS}_{args.reading_name}_val_loader_{band}_{args.elec_type}_fold{args.fold_ind}_percentage_{args.percent}.pkl"
                            with open(os.path.join(data_save_path, val_filename), 'rb') as f:
                                val_loader = dill.load(f)
                            get_predicted_trace(args,vqgan,val_loader,test_name=reading_name,infer=False,perm=False,percent=percent)
